In [1]:
# Parameters
megadescriptor_version = 'T-224'  # 'S-224', 'B-224', 'L-384'
detection = '_detected_manual' # '', _detected', '_detected_manual'
seed = 42
query_ratio = 0.2

In [2]:
# Parameters
query_ratio = 0.2
seed = 0


In [3]:
import torch
import numpy as np
import joblib
import torch.nn as nn
import torch.optim as optim
import random
from collections import defaultdict
import sys
sys.path.append(r'C:\BP\pythonProject1')
from misclassification_utils import show_misclassified

In [4]:
np.random.seed(seed)

# Path to new file
data_path = f"saved_models/{megadescriptor_version}/data{detection}.npz"
encoder_path = f"saved_models/{megadescriptor_version}/label_encoder{detection}.pkl"

# Load everything at once
data = np.load(data_path)

embeddings = data["embeddings"]      # shape (N, D)
labels = data["label_ids"]           # integer labels
# original_labels = data["labels"]     # string labels (optional)

print("Embeddings shape:", embeddings.shape)

# Optional: load encoder if you want inverse_transform
encoder = joblib.load(encoder_path)
names = encoder.inverse_transform(labels)

encoder = joblib.load(encoder_path)
id_to_name = dict(enumerate(encoder.classes_))
name_to_id = {v: k for k, v in id_to_name.items()}


Embeddings shape: (315, 768)


In [5]:
from sklearn.model_selection import train_test_split

# Create an array of original indices to track which embedding each sample came from
original_indices = np.arange(len(embeddings))

X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    embeddings, labels, original_indices, test_size=query_ratio, random_state=seed
)
import torch
from torch.utils.data import TensorDataset, DataLoader

# convert numpy arrays from the train/test split into torch tensors
# use float32 for the embeddings and long for the integer labels
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

In [6]:
import torch
from torch.utils.data import TensorDataset, DataLoader

# convert numpy arrays from the train/test split into torch tensors
# use float32 for the embeddings and long for the integer labels
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)

# optional: keep full dataset tensors for evaluation later
X = torch.tensor(embeddings, dtype=torch.float32)
y = torch.tensor(labels, dtype=torch.long)

# build training dataset and loader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)


In [7]:
class Classifier(nn.Module):
    def __init__(self, input_dim=768, num_classes=10, hidden_dim=256, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_classes)
        )

    def forward(self, x):
        return self.net(x)


In [8]:
# determine number of classes from training labels
num_classes = len(torch.unique(torch.tensor(y_train)))

model = Classifier(input_dim=768, num_classes=num_classes)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(100):
    model.train()
    total_loss = 0
    correct = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct += (outputs.argmax(1) == y_batch).sum().item()

    acc = correct / len(train_dataset)
    print(f"Epoch {epoch}: loss={total_loss:.3f}, acc={acc:.3f}")


Epoch 0: loss=39.090, acc=0.258
Epoch 1: loss=29.492, acc=0.472
Epoch 2: loss=23.984, acc=0.512
Epoch 3: loss=18.548, acc=0.647
Epoch 4: loss=15.150, acc=0.710
Epoch 5: loss=11.696, acc=0.778
Epoch 6: loss=9.595, acc=0.853
Epoch 7: loss=6.943, acc=0.893


Epoch 8: loss=5.104, acc=0.933
Epoch 9: loss=3.746, acc=0.952
Epoch 10: loss=3.634, acc=0.956
Epoch 11: loss=2.195, acc=0.988
Epoch 12: loss=2.097, acc=0.968
Epoch 13: loss=2.061, acc=0.972
Epoch 14: loss=2.491, acc=0.964
Epoch 15: loss=1.241, acc=0.988


Epoch 16: loss=1.083, acc=0.984
Epoch 17: loss=1.004, acc=0.984
Epoch 18: loss=1.097, acc=0.984
Epoch 19: loss=0.959, acc=0.996
Epoch 20: loss=0.663, acc=0.992
Epoch 21: loss=0.657, acc=0.992
Epoch 22: loss=0.667, acc=0.988
Epoch 23: loss=0.899, acc=0.984
Epoch 24: loss=0.335, acc=1.000


Epoch 25: loss=0.397, acc=0.996
Epoch 26: loss=0.482, acc=0.996
Epoch 27: loss=0.659, acc=0.992
Epoch 28: loss=0.892, acc=0.984
Epoch 29: loss=0.787, acc=0.984
Epoch 30: loss=0.578, acc=0.992
Epoch 31: loss=0.599, acc=0.992
Epoch 32: loss=0.510, acc=0.992
Epoch 33: loss=0.385, acc=0.992


Epoch 34: loss=0.411, acc=0.992
Epoch 35: loss=0.233, acc=0.996
Epoch 36: loss=0.396, acc=0.996
Epoch 37: loss=0.987, acc=0.988
Epoch 38: loss=0.393, acc=0.992
Epoch 39: loss=0.380, acc=0.992
Epoch 40: loss=0.249, acc=0.996
Epoch 41: loss=0.410, acc=0.992
Epoch 42: loss=0.282, acc=0.996


...

Epoch 60: loss=0.224, acc=0.992
Epoch 61: loss=0.154, acc=1.000
Epoch 62: loss=0.181, acc=0.996
Epoch 63: loss=0.361, acc=0.992
Epoch 64: loss=0.496, acc=0.984
Epoch 65: loss=0.815, acc=0.992
Epoch 66: loss=0.302, acc=0.996
Epoch 67: loss=0.595, acc=0.988
Epoch 68: loss=0.840, acc=0.984


Epoch 69: loss=2.019, acc=0.964
Epoch 70: loss=0.991, acc=0.980
Epoch 71: loss=0.832, acc=0.992
Epoch 72: loss=0.293, acc=0.992
Epoch 73: loss=0.201, acc=1.000
Epoch 74: loss=0.288, acc=0.992
Epoch 75: loss=0.276, acc=0.992
Epoch 76: loss=0.240, acc=0.988


Epoch 77: loss=0.475, acc=0.988
Epoch 78: loss=0.211, acc=0.992
Epoch 79: loss=0.419, acc=0.992
Epoch 80: loss=0.352, acc=0.992
Epoch 81: loss=0.426, acc=0.992
Epoch 82: loss=0.362, acc=0.984
Epoch 83: loss=0.454, acc=0.988
Epoch 84: loss=0.619, acc=0.988


Epoch 85: loss=0.415, acc=0.988
Epoch 86: loss=0.591, acc=0.980
Epoch 87: loss=0.285, acc=0.996
Epoch 88: loss=0.206, acc=0.996
Epoch 89: loss=0.511, acc=0.992
Epoch 90: loss=0.821, acc=0.992
Epoch 91: loss=0.605, acc=0.992
Epoch 92: loss=0.292, acc=0.992
Epoch 93: loss=0.832, acc=0.976


Epoch 94: loss=0.302, acc=0.996
Epoch 95: loss=0.358, acc=0.992
Epoch 96: loss=0.218, acc=0.996
Epoch 97: loss=0.413, acc=0.988
Epoch 98: loss=0.292, acc=0.996
Epoch 99: loss=0.183, acc=0.992


In [9]:
# Evaluate on training set
model.eval()
with torch.no_grad():
    preds = model(X_train_tensor).argmax(1)
    accuracy = (preds == y_train_tensor).float().mean()
    print("Final train accuracy:", accuracy.item())

    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    print("Final train loss:", loss.item())

Final train accuracy: 0.9960317611694336
Final train loss: 0.005721391178667545


In [10]:
# Check misclassified samples on training set
show_misclassified(y_train_tensor, preds, idx_train, detection, encoder)

Number wrong: 1


Index 249 (Orig 117): Dio\Dio_4.JPG
  True: Dio, Predicted: Brano



## CrossEntropy Loss


In [11]:
# Evaluate on validation set
model.eval()
with torch.no_grad():
    preds_test = model(X_test_tensor).argmax(1)
    accuracy_test = (preds_test == y_test_tensor).float().mean()
    print("Final validation accuracy:", accuracy_test.item())

    outputs_test = model(X_test_tensor)
    loss_test = criterion(outputs_test, y_test_tensor)
    print("Final validation loss:", loss_test.item())

Final validation accuracy: 0.6507936716079712
Final validation loss: 2.5052525997161865


In [12]:
# reuse helper function defined earlier to list misclassified samples on validation set
show_misclassified(y_test_tensor, preds_test, idx_test, detection, encoder)

Number wrong: 22
Index 1 (Orig 314): Zora\Zora_9.JPG
  True: Zora, Predicted: Albin

Index 2 (Orig 150): Edo\Edo_8.JPG
  True: Edo, Predicted: Milos

Index 3 (Orig 158): Eliska\Eliska_5.JPG
  True: Eliska, Predicted: Albin

Index 4 (Orig 108): Brano\Brano_7.JPG
  True: Brano, Predicted: Milos

Index 5 (Orig 275): Roman\Roman_13.JPG
  True: Roman, Predicted: Milos

Index 11 (Orig 261): Milos\Milos_45.JPG
  True: Milos, Predicted: Edo

Index 13 (Orig 241): Milos\Milos_27.JPG
  True: Milos, Predicted: Brano

Index 14 (Orig 152): Eliska\Eliska_1.JPG
  True: Eliska, Predicted: Roman

Index 18 (Orig 22): Albin\Albin_27.JPG
  True: Albin, Predicted: Roman

Index 19 (Orig 310): Zora\Zora_5.JPG
  True: Zora, Predicted: Kiara

Index 20 (Orig 252): Milos\Milos_37.JPG
  True: Milos, Predicted: Roman

Index 23 (Orig 7): Albin\Albin_12.JPG
  True: Albin, Predicted: Benadik

Index 33 (Orig 240): Milos\Milos_26.JPG
  True: Milos, Predicted: Izidor

Index 34 (Orig 124): Edo\Edo_10.JPG
  True: Edo, Pred

## Poznamenanie k výsledkom tréningu

- **Izidor_27** (nočná fotka zozadu) bol nesprávne klasifikovaný ako **Miloš**, ktorý má v datasete veľa obrázkov zozadu, ale aj veľa nočných.
- **Eliška_7** sa pravdepodobne podobá na **Braňa**.
- **Kiara_17** je nočný dobre osvetlený záber zboku s kontrastným zatmeným pozadím, veľmi podobný mnohým zaberom **Romana** s týmito charakteristikami.
- **Izidor_26** je záber zboku s výnimočne zeleným pozadím, nesprávne klasifikovaný ako **Roman**, ktorý má v datasete (v porovnaní s ostatnými) výrazne veľa snímok zboku.
- **Zora_5** bola pre kombináciu sneh + ihličnany klasifikovaná ako **Izidor**, ktorý má v tréningovom sete veľa obrázkov tohto typu.
- **Izidor_37** (nočná fotka + svietiace oči) bol klasifikovaný ako **Miloš**, ktorý má v datasete veľa obrázkov s touto kombináciou.
- **Brano_2** (jesenná fotka) bol nesprávne klasifikovaný ako **Eliška**, u ktorej sú niektoré jesenné obrázky.

In [13]:
result = {
    "megadescriptor_version": megadescriptor_version,
    "dataset_version": detection,
    "seed": seed,
    "query_ratio": query_ratio,
    "accuracy": accuracy_test.item(),
    "loss_function": "CrossEntropyLoss"
}

result

{'megadescriptor_version': 'T-224',
 'dataset_version': '_detected_manual',
 'seed': 0,
 'query_ratio': 0.2,
 'accuracy': 0.6507936716079712,
 'loss_function': 'CrossEntropyLoss'}